# EP2 · IL2.1 — Agente funcional: Asistente Clínico SAPU
**ISY0101 · Ingeniería de Soluciones con IA** — continuación del caso de la EP1 (RAG clínico para SAPU).

Este notebook cubre **IL2.1 / IE1 e IE2**: un agente con **framework específico (LangChain)** que integra tres tipos de herramientas:

| Tipo | Herramienta | Qué hace |
|---|---|---|
| Consulta | `buscar_protocolo` | RAG (FAISS + embeddings) sobre guías/protocolos, devuelve fuente y página |
| Consulta | `consultar_vademecum` | Verifica si un fármaco existe en el botiquín del SAPU |
| Razonamiento | `calcular_dosis` | Cálculo determinista mg/kg y tope máximo (el LLM no hace aritmética) |
| Escritura | `registrar_derivacion` | Genera y guarda una ficha de derivación en archivo (sin datos del paciente) |

⚠️ **Los documentos de abajo son de DEMO** (texto de ejemplo con valores ilustrativos). Para la entrega final, reemplázalos por los PDFs reales del MINSAL/SAPU (ver sección 3).

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai langchain-community faiss-cpu python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata
    for _k in ("LLM_API_KEY","LLM_BASE_URL","LLM_MODEL"):
        try: os.environ[_k] = userdata.get(_k)
        except Exception: pass
    os.environ.setdefault("LLM_BASE_URL","https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL","mistral-small-latest")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm = ChatOpenAI(base_url=os.getenv("LLM_BASE_URL"), api_key=os.getenv("LLM_API_KEY"),
                 model=os.getenv("LLM_MODEL","mistral-small-latest"), temperature=0)
embeddings = OpenAIEmbeddings(base_url=os.getenv("LLM_BASE_URL"), api_key=os.getenv("LLM_API_KEY"),
                 model="mistral-embed", check_embedding_ctx_length=False)
print("✅ LLM y embeddings configurados")

## 3. Base documental (DEMO)
Cada fragmento lleva metadatos `documento`, `pagina` y `tipo`, para exigir trazabilidad (objetivo específico de la EP1).
Para el repo final: cargar PDFs reales con `PyPDFLoader` + `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` y conservar los metadatos.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

DOCS = [
 Document(page_content="DEMO. Crisis asmática moderada pediátrica: administrar salbutamol inhalado con aerocámara, evaluar respuesta a los 20 minutos. Si no hay mejoría, considerar corticoide sistémico y derivación.",
          metadata={"documento":"Guía Clínica Asma <15 años (DEMO)","pagina":12,"tipo":"guia"}),
 Document(page_content="DEMO. Hidrocortisona de urgencia: dosis ilustrativa 5 mg/kg por dosis, máximo 100 mg por dosis.",
          metadata={"documento":"Vademécum SAPU (DEMO)","pagina":4,"tipo":"vademecum"}),
 Document(page_content="DEMO. Salbutamol: presentación inhalador 100 mcg/puff. Disponible en botiquín de urgencia.",
          metadata={"documento":"Vademécum SAPU (DEMO)","pagina":7,"tipo":"vademecum"}),
 Document(page_content="DEMO. Dolor torácico en adulto: derivación inmediata al hospital base si hay dolor opresivo >20 min, compromiso hemodinámico o alteraciones en ECG. Traslado con soporte avanzado.",
          metadata={"documento":"Protocolo de Traslado y Derivación SAPU (DEMO)","pagina":3,"tipo":"derivacion"}),
 Document(page_content="DEMO. Categorización ESI: nivel 1 requiere intervención inmediata que salve la vida; nivel 2 alto riesgo o confusión; niveles 3-5 según recursos esperados.",
          metadata={"documento":"Manual de Categorización ESI (DEMO)","pagina":2,"tipo":"triage"}),
]
vector_db = FAISS.from_documents(DOCS, embeddings)
print(f"✅ {len(DOCS)} fragmentos indexados")

## 4. Herramientas del agente

In [ ]:
import re, json, datetime, pathlib
from langchain_classic.agents import tool

# Guardrail de la EP1: bloquear datos identificables (RUT) antes de llegar al LLM / archivos
RUT = re.compile(r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b")
def contiene_datos_paciente(texto: str) -> bool:
    return bool(RUT.search(texto))

# ---------- CONSULTA 1: RAG ----------
@tool
def buscar_protocolo(consulta: str) -> str:
    """Busca en guías clínicas, protocolos de derivación y manual ESI del SAPU.
    Devuelve fragmentos con documento y página. Úsala para cualquier pregunta clínica o de derivación."""
    try:
        res = vector_db.similarity_search(consulta, k=3)
        if not res: return "SIN_INFORMACION: no hay fragmentos relevantes."
        return "\n".join(f"[{d.metadata['documento']}, p.{d.metadata['pagina']}] {d.page_content}" for d in res)
    except Exception as e:
        return f"Error en la búsqueda: {e}"

# ---------- CONSULTA 2: vademécum (filtro por metadato) ----------
@tool
def consultar_vademecum(farmaco: str) -> str:
    """Consulta el vademécum del SAPU para un fármaco (dosis por kg, tope máximo, disponibilidad).
    Si el fármaco no aparece, informa que NO está en el botiquín."""
    try:
        res = vector_db.similarity_search(farmaco, k=4, filter={"tipo":"vademecum"})
        hits = [d for d in res if farmaco.lower().split()[0] in d.page_content.lower()]
        if not hits: return f"'{farmaco}' NO figura en el vademécum del SAPU. No inventar datos."
        return "\n".join(f"[{d.metadata['documento']}, p.{d.metadata['pagina']}] {d.page_content}" for d in hits)
    except Exception as e:
        return f"Error en la consulta: {e}"

# ---------- RAZONAMIENTO: cálculo determinista ----------
@tool
def calcular_dosis(peso_kg: float, dosis_mg_por_kg: float, max_mg: float) -> str:
    """Calcula la dosis (mg) = peso_kg * dosis_mg_por_kg, aplicando el tope max_mg.
    Los valores dosis_mg_por_kg y max_mg DEBEN venir de consultar_vademecum, nunca inventarse."""
    if peso_kg <= 0 or dosis_mg_por_kg <= 0 or max_mg <= 0:
        return "Error: peso, dosis y máximo deben ser positivos."
    bruta = peso_kg * dosis_mg_por_kg
    final = min(bruta, max_mg)
    nota = " (se aplicó el tope máximo)" if bruta > max_mg else ""
    return f"Dosis calculada: {final:g} mg{nota}. Requiere validación del médico a cargo."

# ---------- ESCRITURA: ficha de derivación ----------
@tool
def registrar_derivacion(motivo: str, criterio_protocolo: str, prioridad_esi: int) -> str:
    """Guarda una ficha de derivación al hospital base en archivo. NO incluir nombre ni RUT del paciente.
    Parámetros: motivo clínico breve, criterio del protocolo (con fuente) y prioridad ESI (1-5)."""
    if contiene_datos_paciente(motivo + criterio_protocolo):
        return "BLOQUEADO: se detectaron datos identificables (RUT). Reformule sin datos del paciente."
    if prioridad_esi not in (1,2,3,4,5):
        return "Error: prioridad_esi debe estar entre 1 y 5."
    try:
        out = pathlib.Path("derivaciones"); out.mkdir(exist_ok=True)
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        f = out / f"derivacion_{ts}.json"
        f.write_text(json.dumps({"fecha":ts,"motivo":motivo,"criterio":criterio_protocolo,
                                 "esi":prioridad_esi,"estado":"pendiente_validacion_medica"},
                                ensure_ascii=False, indent=2), encoding="utf-8")
        return f"Ficha guardada en {f} (pendiente de validación médica)."
    except Exception as e:
        return f"Error al guardar: {e}"

tools = [buscar_protocolo, consultar_vademecum, calcular_dosis, registrar_derivacion]
print("✅ Herramientas:", [t.name for t in tools])

## 5. Agente (LangChain, mismo patrón visto en clase: `create_openai_tools_agent` + `AgentExecutor`)

In [ ]:
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM = (
 "Eres un asistente clínico de apoyo documental para el personal de un SAPU. "
 "Responde SOLO con información devuelta por tus herramientas. Si no hay información, dilo y sugiere consultar al médico a cargo. "
 "Nunca inventes dosis ni protocolos. Para dosis: primero consultar_vademecum, luego calcular_dosis con esos valores. "
 "Cita siempre documento y página. Si el caso cumple criterios de derivación inmediata, usa registrar_derivacion. "
 "No solicites ni uses nombres o RUT de pacientes. Toda salida requiere validación del médico. Lenguaje técnico, breve."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])
agent = create_openai_tools_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=6, handle_parsing_errors=True)

def consultar(q):
    if contiene_datos_paciente(q):
        return "BLOQUEADO: la consulta contiene datos identificables del paciente (Ley 20.584)."
    return executor.invoke({"input": q, "chat_history": []})["output"]

## 6. Pruebas (evidencia para el informe: pegar capturas del `verbose`)

In [ ]:
# P1: consulta + razonamiento (vademécum -> cálculo)
print(consultar("Paciente pediátrico de 15 kg con crisis asmática moderada: ¿dosis de hidrocortisona según vademécum?"))

In [ ]:
# P2: consulta + escritura (derivación)
print(consultar("Adulto con dolor opresivo de 30 minutos. ¿Cumple criterio de derivación? Si corresponde, registra la ficha con ESI 2."))

In [ ]:
# P3: fármaco inexistente (no debe inventar)
print(consultar("¿Cuál es la dosis de zorbalix en el SAPU?"))

In [ ]:
# P4: guardrail de datos sensibles
print(consultar("Paciente Juan Pérez RUT 12.345.678-5 con dolor torácico, ¿qué hago?"))